# 15: Embedding Visualization - Seeing Words in Space

## Making the Invisible Visible

Word embeddings live in high-dimensional space (50-300 dimensions). We can't visualize that directly, but we can project them down to 2D or 3D to see patterns.

### The Web Dev Analogy

This is like a **dashboard visualization** for complex data:
- You have 100 metrics per user
- You reduce to 2D to see clusters
- Similar users appear near each other

We'll use:
- **PCA**: Linear projection, fast, preserves global structure
- **t-SNE**: Non-linear, slower, preserves local neighborhoods

## What You'll Learn
- [ ] Apply PCA and t-SNE for dimensionality reduction
- [ ] Visualize and interpret word embedding clusters
- [ ] Find nearest neighbors in embedding space using cosine similarity

## Connection to Previous Lessons

| What you learned | How it connects here |
|-----------------|---------------------|
| **Lesson 14**: Word2Vec trained embeddings | Now we visualize what those embeddings actually captured |
| **Lesson 13**: Cosine similarity | We use it to find nearest neighbors and verify embedding quality |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from mpl_toolkits.mplot3d import Axes3D
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
np.random.seed(42)

print("Ready to visualize word embeddings!")

## 1. Creating Meaningful Sample Embeddings

Let's create embeddings that simulate what real pre-trained embeddings look like.

In [ ]:
# Create a vocabulary with semantic groups
word_groups = {
    'animals': ['dog', 'cat', 'bird', 'fish', 'horse', 'elephant', 'lion', 'tiger'],
    'fruits': ['apple', 'banana', 'orange', 'grape', 'mango', 'peach', 'berry', 'melon'],
    'countries': ['france', 'germany', 'italy', 'spain', 'japan', 'china', 'brazil', 'india'],
    'colors': ['red', 'blue', 'green', 'yellow', 'purple', 'orange', 'pink', 'brown'],
    'verbs': ['run', 'walk', 'jump', 'swim', 'fly', 'climb', 'dance', 'sing'],
}

# Create embeddings that cluster by group
embedding_dim = 50
embeddings = {}

# Each group has a base vector, words within group are variations
for group_name, words in word_groups.items():
    # Random base vector for the group
    base = np.random.randn(embedding_dim) * 2
    
    for word in words:
        # Add small random variation to base
        variation = np.random.randn(embedding_dim) * 0.3
        embeddings[word] = base + variation

vocab = list(embeddings.keys())
print(f"Vocabulary size: {len(vocab)}")
print(f"Embedding dimension: {embedding_dim}")
print(f"\nWord groups: {list(word_groups.keys())}")

In [ ]:
# Convert to matrix for visualization
embedding_matrix = np.array([embeddings[word] for word in vocab])
print(f"Embedding matrix shape: {embedding_matrix.shape}")

# Create color mapping for groups
colors = []
group_colors = {
    'animals': 'red',
    'fruits': 'green', 
    'countries': 'blue',
    'colors': 'purple',
    'verbs': 'orange'
}

word_to_group = {}
for group, words in word_groups.items():
    for word in words:
        word_to_group[word] = group

colors = [group_colors[word_to_group[word]] for word in vocab]
print("\nColor coding by semantic group ready!")

## 2. PCA: Principal Component Analysis

PCA finds the directions of maximum variance and projects data onto them.

**Pros**: Fast, preserves global distances
**Cons**: Linear, may miss non-linear structure

In [ ]:
# Apply PCA to reduce to 2D
pca = PCA(n_components=2)
embeddings_pca_2d = pca.fit_transform(embedding_matrix)

print(f"Original shape: {embedding_matrix.shape}")
print(f"After PCA: {embeddings_pca_2d.shape}")
print(f"\nVariance explained: {pca.explained_variance_ratio_.sum():.1%}")

In [ ]:
# Visualize PCA 2D
plt.figure(figsize=(14, 10))

# Plot each point
for i, word in enumerate(vocab):
    x, y = embeddings_pca_2d[i]
    plt.scatter(x, y, c=colors[i], s=100, alpha=0.7)
    plt.annotate(word, (x + 0.02, y + 0.02), fontsize=9)

# Add legend
for group, color in group_colors.items():
    plt.scatter([], [], c=color, label=group.capitalize(), s=100)

plt.xlabel('PCA Component 1', fontsize=12)
plt.ylabel('PCA Component 2', fontsize=12)
plt.title('Word Embeddings Visualized with PCA (2D)', fontsize=14)
plt.legend(loc='best', fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Words from the same category cluster together!")

## 3. t-SNE: t-Distributed Stochastic Neighbor Embedding

t-SNE focuses on preserving **local neighborhoods** - points that are close in high-D stay close in 2D.

**Pros**: Great for revealing clusters, handles non-linear relationships
**Cons**: Slower, distances between clusters not meaningful

In [ ]:
# Apply t-SNE
tsne = TSNE(n_components=2, perplexity=10, random_state=42, n_iter=1000)
embeddings_tsne_2d = tsne.fit_transform(embedding_matrix)

print(f"t-SNE complete!")
print(f"Shape: {embeddings_tsne_2d.shape}")

In [ ]:
# Visualize t-SNE 2D
plt.figure(figsize=(14, 10))

# Plot each point
for i, word in enumerate(vocab):
    x, y = embeddings_tsne_2d[i]
    plt.scatter(x, y, c=colors[i], s=100, alpha=0.7)
    plt.annotate(word, (x + 0.5, y + 0.5), fontsize=9)

# Add legend
for group, color in group_colors.items():
    plt.scatter([], [], c=color, label=group.capitalize(), s=100)

plt.xlabel('t-SNE Dimension 1', fontsize=12)
plt.ylabel('t-SNE Dimension 2', fontsize=12)
plt.title('Word Embeddings Visualized with t-SNE (2D)', fontsize=14)
plt.legend(loc='best', fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("t-SNE often shows clearer cluster separation!")

## 4. PCA vs t-SNE: Side by Side

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# PCA
ax1 = axes[0]
for i, word in enumerate(vocab):
    x, y = embeddings_pca_2d[i]
    ax1.scatter(x, y, c=colors[i], s=80, alpha=0.7)
    ax1.annotate(word, (x + 0.02, y + 0.02), fontsize=8)
ax1.set_title('PCA Projection', fontsize=14, fontweight='bold')
ax1.set_xlabel('Component 1')
ax1.set_ylabel('Component 2')
ax1.grid(True, alpha=0.3)

# t-SNE
ax2 = axes[1]
for i, word in enumerate(vocab):
    x, y = embeddings_tsne_2d[i]
    ax2.scatter(x, y, c=colors[i], s=80, alpha=0.7)
    ax2.annotate(word, (x + 0.5, y + 0.5), fontsize=8)
ax2.set_title('t-SNE Projection', fontsize=14, fontweight='bold')
ax2.set_xlabel('Dimension 1')
ax2.set_ylabel('Dimension 2')
ax2.grid(True, alpha=0.3)

# Shared legend
for group, color in group_colors.items():
    ax2.scatter([], [], c=color, label=group.capitalize(), s=80)
ax2.legend(loc='upper right', fontsize=9)

plt.tight_layout()
plt.show()

print("PCA preserves global structure, t-SNE emphasizes local clusters.")

## 5. 3D Visualization

Sometimes 3D gives a better view of the embedding space.

In [ ]:
# 3D PCA
pca_3d = PCA(n_components=3)
embeddings_pca_3d = pca_3d.fit_transform(embedding_matrix)

fig = plt.figure(figsize=(12, 10))
ax = fig.add_subplot(111, projection='3d')

# Plot each point
for i, word in enumerate(vocab):
    x, y, z = embeddings_pca_3d[i]
    ax.scatter(x, y, z, c=colors[i], s=60, alpha=0.7)
    ax.text(x, y, z, word, fontsize=8)

# Add legend
for group, color in group_colors.items():
    ax.scatter([], [], [], c=color, label=group.capitalize(), s=60)

ax.set_xlabel('PC1')
ax.set_ylabel('PC2')
ax.set_zlabel('PC3')
ax.set_title('Word Embeddings in 3D (PCA)', fontsize=14)
ax.legend(loc='upper left')
plt.tight_layout()
plt.show()

print(f"3D explains {pca_3d.explained_variance_ratio_.sum():.1%} of variance")

## 6. Finding Similar Words

The most practical use of embeddings: finding semantically similar words.

In [ ]:
def cosine_similarity(v1, v2):
    """Compute cosine similarity between two vectors."""
    return np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2) + 1e-10)

def find_similar_words(query_word, embeddings, top_k=5):
    """Find most similar words to a query word."""
    if query_word not in embeddings:
        return f"Word '{query_word}' not in vocabulary"
    
    query_vec = embeddings[query_word]
    similarities = []
    
    for word, vec in embeddings.items():
        if word != query_word:
            sim = cosine_similarity(query_vec, vec)
            similarities.append((word, sim))
    
    # Sort by similarity (descending)
    similarities.sort(key=lambda x: x[1], reverse=True)
    return similarities[:top_k]

# Test with different words
test_words = ['dog', 'apple', 'france', 'red', 'run']

for word in test_words:
    print(f"\nWords most similar to '{word}':")
    similar = find_similar_words(word, embeddings)
    for sim_word, score in similar:
        print(f"  {sim_word}: {score:.3f}")

In [ ]:
# Visualize similarity as a heatmap
def compute_similarity_matrix(words, embeddings):
    """Compute pairwise similarity matrix."""
    n = len(words)
    matrix = np.zeros((n, n))
    
    for i, w1 in enumerate(words):
        for j, w2 in enumerate(words):
            matrix[i, j] = cosine_similarity(embeddings[w1], embeddings[w2])
    
    return matrix

# Select a subset for visualization
subset_words = ['dog', 'cat', 'lion', 'apple', 'banana', 'france', 'germany', 'red', 'blue']
sim_matrix = compute_similarity_matrix(subset_words, embeddings)

plt.figure(figsize=(10, 8))
plt.imshow(sim_matrix, cmap='RdYlGn', aspect='auto', vmin=-1, vmax=1)
plt.colorbar(label='Cosine Similarity')
plt.xticks(range(len(subset_words)), subset_words, rotation=45, ha='right')
plt.yticks(range(len(subset_words)), subset_words)
plt.title('Word Similarity Matrix', fontsize=14)
plt.tight_layout()
plt.show()

print("Green = similar, Red = dissimilar")

## 7. Interactive Exploration: Word Neighborhoods

Let's visualize the neighborhood around a specific word.

In [ ]:
def visualize_neighborhood(query_word, embeddings, embeddings_2d, vocab, n_neighbors=10):
    """Visualize a word and its nearest neighbors."""
    # Find similar words
    similar = find_similar_words(query_word, embeddings, top_k=n_neighbors)
    neighbor_words = [query_word] + [w for w, _ in similar]
    
    plt.figure(figsize=(12, 10))
    
    # Plot all words in gray
    for i, word in enumerate(vocab):
        if word not in neighbor_words:
            x, y = embeddings_2d[i]
            plt.scatter(x, y, c='lightgray', s=50, alpha=0.3)
            plt.annotate(word, (x, y), fontsize=7, alpha=0.3)
    
    # Highlight the query word
    query_idx = vocab.index(query_word)
    qx, qy = embeddings_2d[query_idx]
    plt.scatter(qx, qy, c='red', s=200, marker='*', edgecolors='black', linewidth=2)
    plt.annotate(query_word, (qx + 0.02, qy + 0.02), fontsize=12, fontweight='bold', color='red')
    
    # Highlight neighbors
    for word, sim in similar:
        idx = vocab.index(word)
        x, y = embeddings_2d[idx]
        plt.scatter(x, y, c='blue', s=100, alpha=0.7)
        plt.annotate(f"{word}\n({sim:.2f})", (x + 0.02, y + 0.02), fontsize=10, color='blue')
        # Draw line to query
        plt.plot([qx, x], [qy, y], 'b--', alpha=0.3, linewidth=1)
    
    plt.xlabel('Dimension 1', fontsize=12)
    plt.ylabel('Dimension 2', fontsize=12)
    plt.title(f'Neighborhood of "{query_word}"', fontsize=14)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

# Visualize neighborhood for different words
visualize_neighborhood('dog', embeddings, embeddings_tsne_2d, vocab)

In [ ]:
# Try another word
visualize_neighborhood('france', embeddings, embeddings_tsne_2d, vocab)

## 8. Exploring Word Relationships with Vectors

In [ ]:
def word_arithmetic(positive_words, negative_words, embeddings, top_k=5):
    """Perform word arithmetic: sum of positive - sum of negative."""
    result_vec = np.zeros(embedding_dim)
    
    for word in positive_words:
        result_vec += embeddings[word]
    for word in negative_words:
        result_vec -= embeddings[word]
    
    # Find closest words
    exclude = set(positive_words + negative_words)
    similarities = []
    
    for word, vec in embeddings.items():
        if word not in exclude:
            sim = cosine_similarity(result_vec, vec)
            similarities.append((word, sim))
    
    similarities.sort(key=lambda x: x[1], reverse=True)
    return similarities[:top_k]

# Examples
print("Word Arithmetic Examples:")
print("=" * 40)

# dog - cat + lion = ?
print("\ndog - cat + lion = ?")
result = word_arithmetic(['dog', 'lion'], ['cat'], embeddings)
for word, sim in result[:3]:
    print(f"  {word}: {sim:.3f}")

# france - germany + italy = ?
print("\nfrance + italy - germany = ?")
result = word_arithmetic(['france', 'italy'], ['germany'], embeddings)
for word, sim in result[:3]:
    print(f"  {word}: {sim:.3f}")

## 9. Understanding t-SNE Parameters

In [ ]:
# Visualize effect of perplexity
fig, axes = plt.subplots(2, 2, figsize=(14, 14))
perplexities = [5, 10, 20, 30]

for ax, perp in zip(axes.flat, perplexities):
    tsne = TSNE(n_components=2, perplexity=perp, random_state=42, n_iter=1000)
    emb_2d = tsne.fit_transform(embedding_matrix)
    
    for i, word in enumerate(vocab):
        ax.scatter(emb_2d[i, 0], emb_2d[i, 1], c=colors[i], s=60, alpha=0.7)
        ax.annotate(word, (emb_2d[i, 0] + 0.5, emb_2d[i, 1] + 0.5), fontsize=7)
    
    ax.set_title(f'Perplexity = {perp}', fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3)

plt.suptitle('Effect of t-SNE Perplexity on Visualization', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("Lower perplexity = tighter clusters, higher = more global structure")

## 10. Best Practices for Embedding Visualization

In [ ]:
# Summary visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# PCA - When to use
ax1 = axes[0]
ax1.text(0.5, 0.95, 'PCA', ha='center', fontsize=16, fontweight='bold',
        transform=ax1.transAxes)
ax1.text(0.5, 0.85, 'Use When:', ha='center', fontsize=12, fontweight='bold',
        transform=ax1.transAxes, color='green')
pca_points = [
    '- Need fast results',
    '- Want interpretable axes',
    '- Care about global structure',
    '- Distances should be meaningful',
    '- Initial exploration'
]
for i, point in enumerate(pca_points):
    ax1.text(0.1, 0.7 - i*0.1, point, fontsize=11, transform=ax1.transAxes)

ax1.text(0.5, 0.2, 'Limitations:', ha='center', fontsize=12, fontweight='bold',
        transform=ax1.transAxes, color='red')
ax1.text(0.1, 0.1, '- Only linear projections', fontsize=11, transform=ax1.transAxes)
ax1.text(0.1, 0.0, '- May miss complex patterns', fontsize=11, transform=ax1.transAxes)
ax1.axis('off')

# t-SNE - When to use
ax2 = axes[1]
ax2.text(0.5, 0.95, 't-SNE', ha='center', fontsize=16, fontweight='bold',
        transform=ax2.transAxes)
ax2.text(0.5, 0.85, 'Use When:', ha='center', fontsize=12, fontweight='bold',
        transform=ax2.transAxes, color='green')
tsne_points = [
    '- Finding clusters',
    '- Visualizing local neighborhoods',
    '- Revealing hidden structure',
    '- Final presentation graphics',
    '- Complex, non-linear data'
]
for i, point in enumerate(tsne_points):
    ax2.text(0.1, 0.7 - i*0.1, point, fontsize=11, transform=ax2.transAxes)

ax2.text(0.5, 0.2, 'Limitations:', ha='center', fontsize=12, fontweight='bold',
        transform=ax2.transAxes, color='red')
ax2.text(0.1, 0.1, '- Slower computation', fontsize=11, transform=ax2.transAxes)
ax2.text(0.1, 0.0, '- Distances between clusters not meaningful', fontsize=11, transform=ax2.transAxes)
ax2.axis('off')

plt.tight_layout()
plt.show()

## Check Your Understanding

1. What's the difference between PCA and t-SNE?
2. Why can't we directly visualize 50-dimensional embeddings?
3. What does the perplexity parameter control in t-SNE?
4. How do we find similar words using embeddings?
5. When would you use PCA vs t-SNE for visualization?

In [ ]:
# --- Quick Check: PCA vs t-SNE ---
# PCA preserves _____ structure, while t-SNE preserves _____ structure.
# a) local, global
# b) global, local
# c) linear, non-linear
# d) They preserve the same thing

your_answer = None  # Put 'a', 'b', 'c', or 'd'

# --- Check ---
assert your_answer is not None, "Pick an answer!"
assert your_answer == 'b', "PCA keeps global variance (overall spread), t-SNE keeps local neighborhoods (nearby points stay near)."
print("Exercise 1 passed! ✓")

# --- Exercise 2: Dimensionality Reduction Shape ---
# If you have 300-dim embeddings for 1000 words and apply PCA to reduce to 2D,
# what is the shape of the result?

# YOUR CODE HERE:
result_shape = None  # A tuple like (rows, cols)

# --- Check ---
assert result_shape is not None, "What shape would the reduced data be?"
assert result_shape == (1000, 2), f"1000 words × 2 dimensions = (1000, 2), got {result_shape}"
print("Exercise 2 passed! ✓")

print("\n🎉 All exercises passed!")

## Summary

**Visualization Techniques:**
- **PCA**: Fast, linear, preserves global structure
- **t-SNE**: Slower, non-linear, reveals clusters
- Both reduce high-D to 2D/3D for human viewing

**Finding Similar Words:**
- Use cosine similarity between embedding vectors
- Similar words cluster together in embedding space
- Similarity matrices show relationships at a glance

**Key Insights:**
- Semantic categories form natural clusters
- Visualization helps validate embedding quality
- Word arithmetic can be visualized as vector operations

**Next up**: Why sequence order matters in language processing!